In [15]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import pandas as pd
import json
from src.data_process import GenerateAnomaly, PrepareDf
from src.metric import ProbTimeDetection, FalseRate


In [16]:
from pyod.models.abod import ABOD
from pyod.models.ocsvm import OCSVM

In [17]:
# Data folder 
data_folder = "/Users/vuongdai/GitHub/bm/detrend_data/monthly/"

# Data file
data_file = "ts_monthly_values_standardize.csv"

# Time file
data_file_time = "ts_monthly_datetimes.csv"

sensor_names = pd.read_csv(data_folder + data_file, nrows=0, delimiter=",").columns.tolist()
df = pd.read_csv(data_folder + data_file, skiprows=1, delimiter=",", header=None)
df_time = pd.read_csv(data_folder + data_file_time, skiprows=1, delimiter=",", header=None)

df_dict = PrepareDf(df, df_time)

# Anomaly info.
anomaly_info_file = "../detrend_data/anomaly_info/anomaly_info.json"
with open(anomaly_info_file, "r") as f:
    anomaly_info = json.load(f)

In [18]:
# Generate anomalies
df_with_anomaly = GenerateAnomaly(data=df_dict, anomaly_info=anomaly_info, col=[0,1])

In [19]:
# ---------------------------------------------------------------------------
# Detector configuration
# Set DETECTOR_NAME to switch between detectors.
# Supported: "abod", "ocsvm"
# ---------------------------------------------------------------------------
LOOK_BACK_LEN = 19       # sliding-window size, mirrors the original LSTM look_back_len
DETECTOR_NAME = "abod"  # <-- change this to swap detectors


def make_sliding_windows(values: np.ndarray, look_back: int) -> np.ndarray:
    """Convert a 1-D time series into overlapping sliding-window feature rows."""
    values = values.ravel()
    n = len(values)
    if n <= look_back:
        pad = np.full(look_back - n, values[0])
        values = np.concatenate([pad, values])
        n = len(values)
    return np.array([values[i: i + look_back] for i in range(n - look_back)])


def build_detector(train_series: np.ndarray, look_back: int = LOOK_BACK_LEN, name: str = DETECTOR_NAME):
    """
    Fit a pyod detector on sliding windows from the training series.

    Parameters
    ----------
    train_series : array-like 1-D training values
    look_back    : sliding-window width (number of lagged features)
    name         : which detector to use — "abod" or "ocsvm"
    """
    X_train = make_sliding_windows(train_series, look_back)

    if name == "abod":
        detector = ABOD(
            contamination=0.05,  # expected fraction of outliers
            n_neighbors=10,      # neighbours used in angle computation
            method="fast",       # "fast" = FastABOD; "default" = exact
        )
    elif name == "ocsvm":
        detector = OCSVM(
            contamination=0.05,  # expected fraction of outliers
            kernel="rbf",        # kernel: "rbf", "linear", "poly", "sigmoid"
            nu=0.05,             # upper bound on fraction of outliers (≈ contamination)
            gamma="auto",        # kernel coefficient; "auto" = 1/n_features
        )
    else:
        raise ValueError(f"Unknown detector '{name}'. Choose from: 'abod', 'ocsvm'.")

    detector.fit(X_train)
    return detector


# ---------------------------------------------------------------------------
# Train on column index 1 of df (same as original script)
# ---------------------------------------------------------------------------
train_split = 0.4
df_temp = df.iloc[:, [1]]
train_size = int(len(df_temp) * train_split)
train_series = df_temp.values[:train_size]

detector = build_detector(train_series, look_back=LOOK_BACK_LEN)


# ---------------------------------------------------------------------------
# Detection helpers
# ---------------------------------------------------------------------------

def score_series(values: np.ndarray, det: ABOD, look_back: int = LOOK_BACK_LEN) -> np.ndarray:
    """
    Return a per-timestep binary anomaly flag (True = anomaly).
    The first `look_back` timesteps inherit the label of the first window.
    NaNs are imputed via forward-fill then backward-fill before windowing.
    """
    # Impute NaNs: forward-fill then backward-fill to handle leading/trailing NaNs
    values = pd.Series(values.ravel()).ffill().bfill().values
    X = make_sliding_windows(values, look_back)
    labels = det.predict(X).astype(bool)          # 1 = outlier, 0 = inlier
    prefix = np.full(look_back, labels[0])
    return np.concatenate([prefix, labels])        # shape: (len(values),)


def detect_synthetic_anomaly(
    data: dict,
    det: ABOD,
    look_back: int = LOOK_BACK_LEN,
) -> dict:
    anomaly_score = {}
    for ts in data:
        anomaly_score[ts] = {}
        for anomaly_type in data[ts]:
            anomaly_score[ts][anomaly_type] = {}
            for i in data[ts][anomaly_type]:
                anomaly_score[ts][anomaly_type][i] = {}
                for j in data[ts][anomaly_type][i]:
                    series_df = data[ts][anomaly_type][i][j]
                    flags = score_series(series_df.values, det, look_back)
                    anomaly_score[ts][anomaly_type][i][j] = pd.DataFrame(
                        {"value": flags}, index=series_df.index
                    )
    return anomaly_score


def detect_anomaly(
    data: dict,
    det: ABOD,
    look_back: int = LOOK_BACK_LEN,
) -> dict:
    anomaly_score = {}
    for ts in data:
        series_df = data[ts]
        flags = score_series(series_df.values, det, look_back)
        anomaly_score[ts] = pd.DataFrame({"value": flags}, index=series_df.index)
    return anomaly_score


In [ ]:
# ---------------------------------------------------------------------------
# Estimate anomaly scores & evaluate
# ---------------------------------------------------------------------------
anomaly_score_synthetic = detect_synthetic_anomaly(data=df_with_anomaly, det=detector)
anomaly_score_test_set = detect_anomaly(data=df_dict, det=detector)

prob_detection, time_to_detection = ProbTimeDetection(anomaly_score_synthetic, anomaly_info)
false_rate = FalseRate(anomaly_score_test_set)

In [21]:
prob_detection

{0: {'level': {'0.45': 0.7, '0.01': 0.2, '0.1': 0.2},
  'trend': {'0.01': 0.6, '0.05': 0.7, '0.1': 0.7}},
 1: {'level': {'0.45': 0.8, '0.01': 0.1, '0.1': 0.1},
  'trend': {'0.01': 0.6, '0.05': 0.7, '0.1': 0.7}}}

In [22]:
time_to_detection

{0: {'level': {'0.45': Timedelta('304 days 02:24:00'),
   '0.01': Timedelta('12 days 04:48:00'),
   '0.1': Timedelta('12 days 04:48:00')},
  'trend': {'0.01': Timedelta('651 days 00:00:00'),
   '0.05': Timedelta('279 days 16:48:00'),
   '0.1': Timedelta('188 days 12:00:00')}},
 1: {'level': {'0.45': Timedelta('647 days 14:24:00'),
   '0.01': Timedelta('0 days 00:00:00'),
   '0.1': Timedelta('0 days 00:00:00')},
  'trend': {'0.01': Timedelta('839 days 09:36:00'),
   '0.05': Timedelta('355 days 21:36:00'),
   '0.1': Timedelta('246 days 02:24:00')}}}

In [23]:
false_rate

{0: 0.17782375851996107,
 1: 0.5334712755598832,
 2: 4.68075651095494,
 3: 1.2511735600101497,
 4: 0.0,
 5: 0.0,
 6: 0.8804554681552905,
 7: 2.0450727883538633,
 8: 0.0,
 9: 0.0,
 10: 0.0,
 11: 0.0,
 12: 3.506795131845842,
 13: 0.24695740365111563,
 14: 0.0,
 15: 0.7414361837097183,
 16: 0.16666666666666666,
 17: 0.3333333333333333,
 18: 0.0,
 19: 1.8999302207561533,
 20: 1.4365326059375791,
 21: 4.787798527563297,
 22: 0.0,
 23: 0.48993963782696176,
 24: 0.4999315630988229,
 25: 0.0,
 26: 0.10084207620099393,
 27: 3.099575691212702,
 28: 3.293366251610133,
 29: 1.4741975783202,
 30: 1.2694312796208531,
 31: 1.3513057671381936,
 32: 2.9200799474318253,
 33: 4.01417301414581,
 34: 0.0,
 35: 0.08333333333333333,
 36: 4.2738523852385235,
 37: 0.9873674360573924,
 38: 0.0,
 39: 0.0,
 40: 0.9166666666666666,
 41: 2.555749923942805,
 42: 0.8763842746400886,
 43: 0.0,
 44: 0.15890798346747878,
 45: 0.5410382440075411,
 46: 0.19674117963910587,
 47: 0.34429706436843527,
 48: 0.6619845944721341